# Relation to Graph Mapping, Example 1
This example will provide an illustration of the relation to graph mapping and graph based analysis of a real dataset. This dataset is from the SBA. A mapping to the guidelines provided in the [relation to graph mapping](https://github.com/rajivsam/descriptive_analytics/blob/main/examples/graph_from_relations/rel-to-graph-concepts.pdf) is provided. 

## Initial Assessment
The following is done as part of initial assessment
1. Data quality assessment
2. Data analysis to remove data inconsistent with known business conditions
3. Data analysis to remove data with too little support to make valid inference
4. Remediation to fix data with little support by broadening codes that are hierarchical, like industry classification codes and zip codes
5. Defining new attributes as needed
6. Defining the attributes needed for analysis and dropping unnecessary attributes

### Use the utility classes to process the specification for relational to graph mapping
An example of using the relation to heterogeneous graph mapping is provided. The analysis component requires a homogeneous graph. Please see the analysis notebook for an example of relational to homogeneous graph mapping.

In [ ]:
import yaml
fp = "../config/sba_loans/data_processing.yml"

In [ ]:
import sys
sys.path.append('../src')

In [ ]:
from relation_to_graph.mapper.hetero_r2g_mapper import HeteroR2GMapper

In [ ]:
hetr2gm = HeteroR2GMapper(fp)

### A brief summary of the dataset
The small business administration (SBA) makes it possible for small businesses to obtain working capital. They gaurantee borowers, lenders use this guarantee to lend money to business owners. For more information, see [this page](https://www.sba.gov/funding-programs/loans/7a-loans). The SBA reports loan performance. Most loans are paid in full, a small fraction default. We can apply machine learning to determine which loans default. Please refer to the data dictionary in the data folder for detailed description of attributes in the dataset.

In [ ]:
hetr2gm._r2g_cfg["raw_data"]


In [ ]:
fp = "../data/" + hetr2gm._r2g_cfg["raw_data"]["location"]

In [ ]:
import pandas as pd
dfr = pd.read_csv(fp)

In [ ]:
dfr.head()

In [ ]:
attr_list = dfr.columns.tolist()
pd.DataFrame({"attribute": attr_list})

In [ ]:
dfr["AsofDate"] = pd.to_datetime(dfr.AsOfDate)

In [ ]:
len(dfr["BankFDICNumber"].unique())

In [ ]:
len(dfr["BorrName"].unique())

In [ ]:
dfr["LoanStatus"].unique()

In [ ]:
len(dfr["NaicsCode"].unique())

### Exclude small number of loans with unknown business implication
There are about 3 loans which have the first disbursement date that is later than the date the loan was pain in full. This is either a data issue or, more likely, a business condition that is not

In [ ]:
dfr["FirstDisbursementDate"] = pd.to_datetime(dfr["FirstDisbursementDate"])
dfr["PaidInFullDate"] = pd.to_datetime(dfr["PaidInFullDate"])
pna = (dfr["FirstDisbursementDate"].isna()) | (dfr["FirstDisbursementDate"] > dfr["PaidInFullDate"])

to_exclude = (dfr.LoanStatus == "EXEMPT") | (dfr.LoanStatus == "CANCLD") | (dfr.LoanStatus == "COMMIT") | pna
dfr = dfr[~ to_exclude]

### Compute Initial Imbalance

In [ ]:
num_PIF = dfr[dfr.LoanStatus == "PIF"].shape[0]
num_CHGOFF = dfr[dfr.LoanStatus == "CHGOFF"].shape[0]

In [ ]:
dfr["LoanStatus"].value_counts()

In [ ]:
pct_chgoff = (num_CHGOFF/dfr.shape[0])*100
pct_pif = (num_PIF/dfr.shape[0])* 100
print(f" percent charged off is {pct_chgoff:.2f} %, percent paid in full {pct_pif:.2f}%")

In [ ]:
from datetime import datetime

def diff_month(row):
    if row["LoanStatus"] == "PIF":
        return (row["PaidInFullDate"].year - row["FirstDisbursementDate"].year) * 12 + row["PaidInFullDate"].month - row["FirstDisbursementDate"].month
    return (row["AsofDate"].year - row["FirstDisbursementDate"].year) * 12 + row["AsofDate"].month - row["FirstDisbursementDate"].month

### Create an Identifier for the Loan
A loan is one of the entity abstractions. This does not have an identifier, so we create one.

In [ ]:
dfr["LoanID"] = dfr.apply(lambda row: "Loan-" + str(row.name), axis=1)

In [ ]:
dfr["LoanID"]

### Create an attribute for number of Payments
The number of payments made with the loan is a derived attribute.

In [ ]:
dfr["NumPmtsMade"] = dfr.apply(diff_month, axis=1)

In [ ]:
dfr["NumPmtsMade"].describe()

In [ ]:
entities_spec = hetr2gm.get_entities()

In [ ]:
entities_spec

In [ ]:
dfr = dfr.reset_index(drop=True)

In [ ]:
dfr

In [ ]:
# # parse configuration
# entity_dict = {}
# for entity_desc in entities_spec:
#     entity_name = [*entity_desc][0]
#     entity = entity_desc[entity_name]
#     if entity_name not in entity_dict:
#         entity_dict[entity_name] = []
#     for k, v in entity.items():
#         for adict in v:
#             entity_dict[entity_name].append(adict["name"])

    

In [ ]:
alist = []
for key, value in entities_spec.items():
    for v in value:
        alist.append(v)

In [ ]:
dfr = dfr[alist]

In [ ]:
dfr

In [ ]:
dfr.LoanStatus.value_counts()

In [ ]:
dfr.dtypes

### Fix Missing Values

In [ ]:
def bad_FDICNumber(row):

    if pd.isna(row["BankFDICNumber"]):
        idstr = row["BankName"][:10] + "-" + str(row["BankZip"])
        row["BankFDICNumber"] = idstr
    if isinstance(row["BankFDICNumber"], float):
        row["BankFDICNumber"] = int(row["BankFDICNumber"])
    return row["BankFDICNumber"]
dfr["BankFDICNumber"] = dfr.apply(bad_FDICNumber, axis=1)

In [ ]:
dfr.isna().sum().sum()

In [ ]:
null_FDIC = dfr.BankFDICNumber.isna()
dfr[null_FDIC]

### Correct Data Types

In [ ]:
dfr.loc[:, "BankFDICNumber"] = dfr["BankFDICNumber"].astype(str)
dfr.loc[:, "BankZip"] = dfr["BankZip"].astype(str)
dfr.loc[:, "BorrZip"] = dfr["BorrZip"].astype(str)
dfr.loc[:, "NaicsCode"] = dfr["NaicsCode"].astype(str)

In [ ]:
dfr.dtypes

###  Fix Support for Codes
Some of the Zip Codes and NAICS codes have insufficient support. We can fix this by considering only the first few characters of the code. These codes are hierarchical, so doing this merges hierarchically

In [ ]:
dfr["BorrZip"] = dfr.BorrZip.str[:2] + 3*"X" 

### Verify Support
Verify the zip codes and NAICS codes have value counts of at least 5 per category

In [ ]:
dfr["BorrZip"].value_counts()

### Fix Support for Cities
Many Borrower and Lender cities have only one or two entries. Collectively, an insufficient support category is created and these loans are bucketed there.

In [ ]:
dfr["BankCity"].value_counts()

In [ ]:
dfr["BorrCity"].value_counts()

In [ ]:
dfr["BankZip"] = dfr.BankZip.str[:3] + 2*"X" 
dfr["BankZip"].value_counts()

In [ ]:
insuff_supp_bank_city = [index for index, value in dfr["BankCity"].value_counts().items() if value < 5]
insuff_supp_borr_city = [index for index, value in dfr["BorrCity"].value_counts().items() if value < 5]

In [ ]:
recode_bank_city = dfr["BankCity"].isin(insuff_supp_bank_city)

recode_borr_city = dfr["BorrCity"].isin(insuff_supp_borr_city)



In [ ]:
dfr.loc[recode_borr_city, "BorrCity"] = "XXXX"
dfr.loc[recode_bank_city, "BankCity"] = "XXXX"

In [ ]:
dfr["BorrCity"].value_counts()

In [ ]:
dfr["BankCity"].value_counts()

In [ ]:
dfr["BankZip"].value_counts()

In [ ]:
drop_insuff_supp = dfr.BankZip == "51XXX"
dfr = dfr[~drop_insuff_supp]

In [ ]:
dfr.loc[:, "NaicsCode"] = dfr.NaicsCode.str[:2] + 4*"X" 

In [ ]:
dfr.NaicsCode.value_counts()

In [ ]:
insuff_supp_borr_zip = [index for index, value in dfr["BorrZip"].value_counts().items() if value < 5]

In [ ]:
insuff_supp_borr_zip

In [ ]:
fp = "../data/sba_loans_prepared/sba_loans_stage1.csv"
dfr.to_csv(fp, index=False)